In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("day-28")
    
    # Use AWS profile credentials from ~/.aws/credentials
    # No access key or secret key stored in code
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "com.amazonaws.auth.profile.ProfileCredentialsProvider"
    )
    
    # S3 connector packages
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.4.2,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.780"
    )
    
    .getOrCreate()
)

:: loading settings :: url = jar:file:/opt/homebrew/Cellar/apache-spark/4.1.1/libexec/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/rahulsinghrana/.ivy2.5.2/cache
The jars for the packages stored in: /Users/rahulsinghrana/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a6e28d01-1cd0-409d-8b6c-4a1c9132a838;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.4.2 in central
	found software.amazon.awssdk#bundle;2.29.52 in central
	found software.amazon.s3.analyticsaccelerator#analyticsaccelerator-s3;1.2.1 in central
	found org.wildfly.openssl#wildfly-openssl;2.1.4.Final in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.780 in central
:: resolution report :: resolve 137ms :: artifacts dl 5ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.780 from central in [default]
	org.ap

In [2]:
customers_df=spark.read.\
    option('header','true').\
        option('inferSchema','true').\
            csv('s3a://pyspark-30-days-rahul-2026/data/customers.csv')
orders_df=spark.read.\
    option('header','true').\
        option('inferSchema','true').\
            csv('s3a://pyspark-30-days-rahul-2026/data/orders.csv')
products_df=spark.read.\
    option('header','true').\
        option('inferSchema','true').\
            csv('s3a://pyspark-30-days-rahul-2026/data/products.csv')

26/08/12 14:57:37 WARN CredentialProviderListFactory: Credentials option fs.s3a.aws.credentials.provider contains AWS v1 SDK entry com.amazonaws.auth.profile.ProfileCredentialsProvider; mapping to software.amazon.awssdk.auth.credentials.ProfileCredentialsProvider
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


**Problem 1** | Hard

Customer retention by segment

A customer is "retained" in a month if they placed at least one order in both that month and the previous month. For each segment, calculate the retention rate (% of customers retained) for each month-to-month transition present in the data.

In [5]:
# Step 1 — distinct customer-month activity
activity = orders_df.withColumn("year_month", F.date_format("order_date", "yyyy-MM")) \
    .select("customer_id", "year_month").distinct()

# Step 2 — generate the "previous month" for each activity row
activity_prev = activity.withColumn(
    "prev_month",
    F.date_format(F.add_months(F.to_date(F.concat(F.col("year_month"), F.lit("-01"))), -1), "yyyy-MM")
)

# Step 3 — self join: was this customer active in prev_month too?
a = activity.alias("a")
b = activity_prev.alias("b")

retained = b.join(
    a,
    (F.col("a.customer_id") == F.col("b.customer_id")) & (F.col("a.year_month") == F.col("b.prev_month")),
    how="left"
).select(
    F.col("b.customer_id").alias("customer_id"),
    F.col("b.year_month").alias("year_month"),
    F.col("a.customer_id").isNotNull().alias("is_retained")
)

# Step 4 — join with customers for segment
retained_with_segment = retained.join(
    customers_df.select("customer_id", "segment"), on="customer_id", how="inner"
)

# Step 5 — retention rate per segment + month
retained_with_segment.groupBy("segment", "year_month").agg(
    F.count("customer_id").alias("active_customers"),
    F.round(100.0 * F.sum(F.col("is_retained").cast("int")) / F.count("customer_id"), 1).alias("retention_pct")
).orderBy("segment", "year_month").show(30)


+----------+----------+----------------+-------------+
|   segment|year_month|active_customers|retention_pct|
+----------+----------+----------------+-------------+
|Enterprise|   2023-01|               4|          0.0|
|Enterprise|   2023-02|               4|          0.0|
|Enterprise|   2023-03|               4|          0.0|
|Enterprise|   2023-04|               4|          0.0|
|Enterprise|   2023-05|               3|          0.0|
|Enterprise|   2023-06|               4|          0.0|
|Enterprise|   2023-07|               3|          0.0|
|Enterprise|   2023-08|               4|          0.0|
|Enterprise|   2023-09|               4|          0.0|
|Enterprise|   2023-10|               3|          0.0|
|       SMB|   2023-01|               4|          0.0|
|       SMB|   2023-02|               3|          0.0|
|       SMB|   2023-03|               4|          0.0|
|       SMB|   2023-04|               3|          0.0|
|       SMB|   2023-05|               4|          0.0|
|       SM

**Problem 2** | Hard

Build a product affinity pair list

For customers who have ordered more than one distinct product, find all pairs of products they have both purchased. Return the pair (alphabetically ordered to avoid duplicates) and how many customers purchased both. Sort by customer count descending.

In [6]:
customer_products = orders_df.select("customer_id", "product_id").distinct()

a = customer_products.alias("a")
b = customer_products.alias("b")

pairs = a.join(
    b,
    (F.col("a.customer_id") == F.col("b.customer_id")) & (F.col("a.product_id") < F.col("b.product_id")),
    how="inner"
).select(
    F.col("a.customer_id").alias("customer_id"),
    F.col("a.product_id").alias("product_a"),
    F.col("b.product_id").alias("product_b")
)

pairs.groupBy("product_a", "product_b").agg(
    F.countDistinct("customer_id").alias("customer_count")
).orderBy(F.col("customer_count").desc()).show(15)

+---------+---------+--------------+
|product_a|product_b|customer_count|
+---------+---------+--------------+
|     P005|     P006|             5|
|     P001|     P003|             5|
|     P005|     P014|             5|
|     P001|     P006|             5|
|     P001|     P002|             4|
|     P005|     P010|             4|
|     P006|     P010|             4|
|     P004|     P007|             3|
|     P002|     P003|             3|
|     P001|     P005|             3|
|     P001|     P017|             3|
|     P002|     P008|             3|
|     P001|     P007|             3|
|     P001|     P014|             3|
|     P010|     P014|             3|
+---------+---------+--------------+
only showing top 15 rows


**Problem 3** | Hard

Rolling 3-order average price per customer

For each customer, ordered by order_date, calculate a rolling average of unit_price over the current order and the 2 preceding orders (3-order rolling window). For customers with fewer than 3 orders so far, average over however many exist.

In [7]:
rolling_window = Window.partitionBy("customer_id").orderBy("order_date").rowsBetween(-2, 0)

orders_df.withColumn(
    "rolling_avg_3", F.round(F.avg("unit_price").over(rolling_window), 2)
).select("customer_id", "order_date", "unit_price", "rolling_avg_3") \
 .orderBy("customer_id", "order_date") \
 .show(15)

+-----------+----------+----------+-------------+
|customer_id|order_date|unit_price|rolling_avg_3|
+-----------+----------+----------+-------------+
|       C001|2023-01-05|   1299.99|      1299.99|
|       C001|2023-03-03|     89.99|       694.99|
|       C001|2023-05-17|    109.99|       499.99|
|       C001|2023-08-01|    449.99|       216.66|
|       C001|2023-10-16|    349.99|       303.32|
|       C002|2023-01-07|    449.99|       449.99|
|       C002|2023-03-06|     59.99|       254.99|
|       C002|2023-05-20|     79.99|       196.66|
|       C002|2023-08-04|   1299.99|       479.99|
|       C002|2023-10-19|     89.99|       489.99|
|       C003|2023-01-10|    349.99|       349.99|
|       C003|2023-03-09|     79.99|       214.99|
|       C003|2023-05-23|     29.99|       153.32|
|       C003|2023-08-07|     49.99|        53.32|
|       C003|2023-10-22|    699.99|       259.99|
+-----------+----------+----------+-------------+
only showing top 15 rows


**Problem 4** | Hard

Identify "at risk" customers

A customer is "at risk" if: their average order value over their last 3 orders is lower than their average order value over their first 3 orders, AND they have placed at least 5 orders total. List these customers with both averages shown.

In [8]:
w_asc  = Window.partitionBy("customer_id").orderBy("order_date")
w_desc = Window.partitionBy("customer_id").orderBy(F.col("order_date").desc())

ranked = orders_df.withColumn("rn_asc", F.row_number().over(w_asc)) \
                   .withColumn("rn_desc", F.row_number().over(w_desc))

# Total orders per customer
order_counts = orders_df.groupBy("customer_id").agg(F.count("order_id").alias("total_orders"))

first_3_avg = ranked.filter(F.col("rn_asc") <= 3) \
    .groupBy("customer_id").agg(F.round(F.avg("unit_price"), 2).alias("first_3_avg"))

last_3_avg = ranked.filter(F.col("rn_desc") <= 3) \
    .groupBy("customer_id").agg(F.round(F.avg("unit_price"), 2).alias("last_3_avg"))

at_risk = first_3_avg.join(last_3_avg, on="customer_id", how="inner") \
    .join(order_counts, on="customer_id", how="inner") \
    .filter((F.col("last_3_avg") < F.col("first_3_avg")) & (F.col("total_orders") >= 5))

at_risk.orderBy(F.col("total_orders").desc()).show()


+-----------+-----------+----------+------------+
|customer_id|first_3_avg|last_3_avg|total_orders|
+-----------+-----------+----------+------------+
|       C001|     499.99|    303.32|           5|
|       C004|     513.32|     73.32|           5|
+-----------+-----------+----------+------------+



In [9]:
spark.stop()